# 2.3 Подбор и загрузка источников данных

Используются четыре типа источников: реляционная таблица CSV, иерархические XML,
JSON API погоды и неструктурированные научные аннотации.
Все исходные выборки ограничены 100 записями на источник.


In [2]:
import base64
import csv
import hashlib
import html
import io
import json
import re
import sqlite3
import time
import zlib
from contextlib import contextmanager, closing
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML, FileLink

LIMIT = 100
TIMINGS = []
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.precision', 3)

@contextmanager
def measured(stage):
    start = time.perf_counter()
    yield
    TIMINGS.append({'Этап': stage, 'Время, мс': (time.perf_counter() - start) * 1000})

def read_sources(raw):
    air = pd.concat([pd.read_csv(io.StringIO(raw[f'{s}.csv']), nrows=48)
                     for s in ['Aotizhongxin', 'Changping']], ignore_index=True)
    air['timestamp'] = pd.to_datetime(air[['year', 'month', 'day', 'hour']]).dt.tz_localize('Asia/Shanghai')
    air['daypart'] = pd.cut(air['hour'], [-1, 5, 11, 17, 23], labels=['Ночь', 'Утро', 'День', 'Вечер'])
    # daypart вычисляется из исходного часа; station и wd взяты из CSV.
    weather_obj = json.loads(raw['weather.json'])
    weather = pd.DataFrame(weather_obj['hourly'])
    weather = weather.rename(columns={'time': 'timestamp'})
    weather['timestamp'] = pd.to_datetime(weather['timestamp']).dt.tz_localize('Asia/Shanghai')
    weather['city'] = 'Beijing'
    weather['weather_group'] = np.select(
        [weather.weather_code.le(1), weather.weather_code.le(3), weather.weather_code.le(48)],
        ['Ясно', 'Облачно', 'Туман'], default='Осадки')
    sectors = np.array(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    weather['wind_sector'] = sectors[((weather.wind_direction_10m.fillna(0) + 22.5) // 45).astype(int) % 8]
    weather.loc[weather.wind_direction_10m.isna(), 'wind_sector'] = pd.NA

    ns = {'wb': 'http://www.worldbank.org'}
    root = ET.fromstring(raw['indicators.xml'])
    if int(root.attrib.get('pages', 1)) != 1:
        raise ValueError('XML содержит несколько страниц; выборку нужно сузить.')
    indicator_rows = []
    for node in root.findall('wb:data', ns):
        value = node.findtext('wb:value', default='', namespaces=ns)
        indicator_rows.append({
            'country_code': node.findtext('wb:countryiso3code', namespaces=ns),
            'country': node.findtext('wb:country', namespaces=ns),
            'year': int(node.findtext('wb:date', namespaces=ns)),
            'indicator': node.find('wb:indicator', ns).attrib['id'],
            'value': float(value) if value else np.nan,
        })
    wb_long = pd.DataFrame(indicator_rows)
    country_rows = []
    for node in ET.fromstring(raw['countries.xml']).findall('wb:country', ns):
        country_rows.append({
            'country_code': node.attrib['id'],
            'region': node.findtext('wb:region', namespaces=ns),
            'income_group_current': node.findtext('wb:incomeLevel', namespaces=ns),
        })
    countries = pd.DataFrame(country_rows)
    wb = wb_long.pivot(index=['country_code', 'country', 'year'], columns='indicator', values='value').reset_index()
    wb.columns.name = None
    wb = wb.rename(columns={'EN.ATM.PM25.MC.M3': 'annual_pm25', 'SP.POP.TOTL': 'population',
                            'SP.URB.TOTL.IN.ZS': 'urban_pct'})
    wb = wb.merge(countries, on='country_code', validate='many_to_one')

    docs = []
    for item in json.loads(raw['abstracts.json'])['resultList']['result'][:8]:
        text = html.unescape(re.sub(r'<[^>]+>', ' ', item.get('abstractText', '')))
        text = re.sub(r'\s+', ' ', text).strip()
        pubtypes = item.get('pubTypeList', {}).get('pubType', [])
        topic = ('Здоровье' if re.search(r'mortality|cardio|biomarker|disease|oxidative|mitochondri', text, re.I)
                 else 'Оценка загрязнения')
        docs.append({
            'doc_id': f"{item['source']}:{item['id']}", 'title': item['title'], 'text': text,
            'journal': item.get('journalInfo', {}).get('journal', {}).get('title', 'Не указан'),
            'language': item.get('language', 'Не указан'), 'topic': topic,
            'publication_type': 'Обзор' if any('review' in s.lower() for s in pubtypes) else 'Исследование',
            'publication_year': int(item['pubYear']), 'char_count': len(text),
            'word_count': len(re.findall(r"[A-Za-z]+(?:[-'][A-Za-z]+)*", text)),
            'author_count': len(item.get('authorList', {}).get('author', [])),
            'country_code': 'CHN', 'city': 'Beijing',
            'url': f"https://europepmc.org/article/{item['source']}/{item['id']}",
        })
    docs = pd.DataFrame(docs)
    counts = {'CSV': len(air), 'JSON API': len(weather),
              'XML': len(wb_long) + len(countries), 'Тексты': len(docs)}
    assert all(0 < n <= LIMIT for n in counts.values()), counts
    return air, weather, wb, docs, wb_long, countries, counts

def clean_merge(air, weather, wb):
    air = air.copy()
    pollutants = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
    for column in pollutants:
        air[column] = pd.to_numeric(air[column], errors='coerce')
        air.loc[air[column] < 0, column] = np.nan
    air = air.drop_duplicates(['station', 'timestamp']).sort_values(['timestamp', 'station'])
    air['wind_group'] = pd.cut(air.WSPM, [-np.inf, 2, 4, np.inf], right=False,
                               labels=['Менее 2 м/с', 'От 2 до 4 м/с', 'От 4 м/с'])
    air['country_code'] = 'CHN'
    air['city'] = 'Beijing'
    weather = weather.drop_duplicates(['city', 'timestamp'])
    joined = air.merge(weather.drop(columns='city'), on='timestamp', how='left',
                       validate='many_to_one', indicator='weather_match')
    joined = joined.merge(wb.drop(columns='country'), on=['country_code', 'year'], how='left',
                           validate='many_to_one', indicator='xml_match')
    assert len(joined) == len(air) <= LIMIT
    return joined

def vectorize_documents(docs):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(stop_words='english', max_features=100,
                                ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    matrix = vectorizer.fit_transform(docs.text)
    terms = vectorizer.get_feature_names_out()
    weights = []
    for i, doc_id in enumerate(docs.doc_id):
        values = matrix[i].toarray().ravel()
        for j in values.argsort()[-8:][::-1]:
            if values[j] > 0:
                weights.append({'doc_id': doc_id, 'term': terms[j], 'weight': float(values[j])})
    return vectorizer, matrix, pd.DataFrame(weights)

def profile(frame, name):
    print(f'{name}: {frame.shape[0]} строк, {frame.shape[1]} столбцов')
    return pd.DataFrame({'Тип': frame.dtypes.astype(str), 'Пропуски': frame.isna().sum(),
                         'Доля пропусков, %': frame.isna().mean() * 100,
                         'Различных значений': frame.nunique(dropna=True)})

FIELD_INFO = {
    'No': ('Номер записи внутри CSV; не глобальный ключ', 'номер', 'Идентификатор', 'номинальная', 'Контроль источника'),
    'year': ('Год измерения', 'год', 'Временной ключ', 'интервальная', 'Связь с XML'),
    'month': ('Месяц измерения', 'месяц', 'Время', 'циклическая', 'Границы выборки'),
    'day': ('День месяца', 'день', 'Время', 'интервальная', 'Сравнение дат'),
    'hour': ('Местный час измерения', 'час', 'Время', 'циклическая', 'Суточный профиль'),
    'PM2.5': ('Концентрация частиц до 2,5 мкм', 'мкг/м³', 'Основной отклик', 'отношений', 'Проверка гипотезы'),
    'PM10': ('Концентрация частиц до 10 мкм', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Сравнение загрязнителей'),
    'NO2': ('Диоксид азота', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Сравнение загрязнителей'),
    'SO2': ('Диоксид серы', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Контроль состава воздуха'),
    'CO': ('Оксид углерода', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Контроль состава воздуха'),
    'O3': ('Озон', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Контроль состава воздуха'),
    'TEMP': ('Температура у станции из UCI', '°C', 'Фактор', 'интервальная', 'Связь с погодой'),
    'PRES': ('Давление у станции из UCI', 'гПа', 'Фактор', 'отношений', 'Погодный фон'),
    'DEWP': ('Температура точки росы у станции', '°C', 'Фактор', 'интервальная', 'Погодный фон'),
    'RAIN': ('Осадки у станции из UCI', 'мм', 'Фактор', 'отношений', 'Возможное вымывание частиц'),
    'wd': ('Направление ветра у станции из UCI', 'румб', 'Категория', 'номинальная', 'Перенос загрязнения'),
    'WSPM': ('Скорость ветра у станции из UCI', 'м/с', 'Фактор', 'отношений', 'Связь со слабым ветром'),
    'station': ('Название станции', 'название', 'Пространственный ключ', 'номинальная', 'Сравнение территорий'),
    'timestamp': ('Дата и время Asia/Shanghai', 'UTC+08:00', 'Временной ключ', 'интервальная', 'Соединение по часу'),
    'daypart': ('Период суток, вычислен из часа', 'четыре периода', 'Производная категория', 'номинальная', 'Суточный профиль'),
    'city': ('Город наблюдения или тема текста', 'название', 'Пространственный ключ', 'номинальная', 'География'),
    'country_code': ('Код страны ISO3', 'код', 'Ключ', 'номинальная', 'Связь источников'),
    'country': ('Название страны', 'название', 'Категория', 'номинальная', 'Страновой контекст'),
    'temperature_2m': ('Температура ERA5 на высоте 2 м', '°C', 'Внешний фактор', 'интервальная', 'Дополняет станционную погоду'),
    'relative_humidity_2m': ('Относительная влажность ERA5', '%', 'Внешний фактор', 'отношений', 'Дополняет станционную погоду'),
    'pressure_msl': ('Давление ERA5 на уровне моря', 'гПа', 'Внешний фактор', 'отношений', 'Погодный фон'),
    'precipitation': ('Осадки ERA5 за предшествующий час', 'мм', 'Внешний фактор', 'отношений', 'Возможное вымывание частиц'),
    'wind_speed_10m': ('Скорость ветра ERA5 на высоте 10 м', 'м/с', 'Внешний фактор', 'отношений', 'Сопоставление со станцией'),
    'wind_direction_10m': ('Направление ветра ERA5', 'градус', 'Внешний фактор', 'циклическая', 'Перенос загрязнения'),
    'weather_code': ('Код погоды WMO из API', 'код', 'Категория в числовой записи', 'номинальная', 'Группировка погоды'),
    'weather_group': ('Группа погоды, вычислена из WMO', 'категория', 'Производная категория', 'номинальная', 'Фильтр погоды'),
    'wind_sector': ('Восемь секторов направления ветра ERA5', 'румб', 'Производная категория', 'номинальная', 'Фильтр ветра'),
    'wind_group': ('Группа скорости ветра UCI', 'три группы', 'Производная категория', 'порядковая', 'Сравнение слабого и сильного ветра'),
    'annual_pm25': ('Средняя годовая экспозиция населения страны к PM2.5', 'мкг/м³', 'Контекст', 'отношений', 'Не почасовой городской показатель'),
    'population': ('Численность населения страны', 'человек', 'Контекст', 'отношений', 'Масштаб территории'),
    'urban_pct': ('Доля городского населения страны', '%', 'Контекст', 'отношений', 'Фон урбанизации'),
    'region': ('Регион по классификации Всемирного банка', 'название', 'Категория', 'номинальная', 'Страновой контекст'),
    'income_group_current': ('Группа дохода на дату запроса, не историческая', 'название', 'Категория', 'порядковая', 'Описание источника, не фактор 2013 года'),
    'indicator': ('Код показателя Всемирного банка', 'код', 'Ключ', 'номинальная', 'Разворот XML в таблицу'),
    'value': ('Значение; единица зависит от кода indicator', 'согласно indicator', 'Показатель', 'отношений', 'Разворот XML в таблицу'),
    'doc_id': ('Идентификатор публикации Europe PMC', 'код', 'Ключ', 'номинальная', 'Исключение дублей'),
    'title': ('Название публикации', 'текст', 'Описание', 'текстовая', 'Проверка релевантности'),
    'text': ('Аннотация, очищенная от HTML', 'текст', 'Неструктурированный признак', 'текстовая', 'Аннотация и векторизация'),
    'journal': ('Научный журнал', 'название', 'Категория', 'номинальная', 'Происхождение текста'),
    'language': ('Язык аннотации', 'код', 'Категория', 'номинальная', 'Выбор обработки текста'),
    'topic': ('Тема по явному правилу ключевых слов', 'категория', 'Производная аннотация', 'номинальная', 'Группировка текстов'),
    'publication_type': ('Тип публикации, укрупнён из метаданных', 'категория', 'Производная категория', 'номинальная', 'Группировка текстов'),
    'publication_year': ('Год публикации, не год измерений статьи', 'год', 'Время', 'интервальная', 'Библиографический контекст'),
    'char_count': ('Число символов аннотации после очистки', 'символ', 'Производный показатель', 'отношений', 'Объём текста'),
    'word_count': ('Число слов по регулярному выражению', 'слово', 'Производный показатель', 'отношений', 'Объём текста'),
    'author_count': ('Число авторов в ответе API', 'автор', 'Метаданные', 'отношений', 'Описание публикации'),
    'url': ('Ссылка на исходную публикацию', 'URL', 'Происхождение', 'номинальная', 'Проверка источника'),
    'weather_match': ('Результат соединения с погодой', 'категория', 'Контроль', 'номинальная', 'Полнота слияния'),
    'xml_match': ('Результат соединения со страновым контекстом', 'категория', 'Контроль', 'номинальная', 'Полнота слияния'),
}

def dictionary(frame):
    rows = []
    for column in frame.columns:
        meaning, unit, role, scale, purpose = FIELD_INFO[column]
        rows.append({'Поле': column, 'Смысл': meaning, 'Единица': unit,
                     'Роль': role, 'Шкала': scale, 'Для анализа': purpose})
    return pd.DataFrame(rows)

def stage_result(joined):
    mean = joined.groupby('station')['PM2.5'].mean().sort_values()
    daily = joined.groupby(joined.timestamp.dt.date)['PM2.5'].mean()
    corr = joined[['PM2.5', 'WSPM']].corr(method='spearman').iloc[0, 1]
    return {'station_means': mean.to_dict(), 'day_means': {str(k): v for k, v in daily.items()},
            'spearman_wind': float(corr)}


In [3]:
# Снимок исходных ответов с URL, датой получения и ограничением числа записей.
# Сжатие уменьшает размер файла; искусственные измерения не добавляются.
RAW = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
with measured('Чтение CSV, JSON, XML и текстов'):
    air, weather, wb, docs, wb_long, countries, counts = read_sources(RAW)
print('Записей по источникам:', counts)


Записей по источникам: {'CSV': 96, 'JSON API': 48, 'XML': 30, 'Тексты': 8}


## Источники и происхождение

- [UCI Beijing Multi-Site Air Quality](https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data), DOI 10.24432/C5RK5G. Табличные измерения из CSV, лицензия CC BY 4.0. Для загрузки первых строк использовано [публичное зеркало CSV](https://github.com/KM-drago/Beijing-Multi-Site-Air-Quality-Data). Зеркало обозначено отдельно от первоисточника; полное побайтовое сравнение с архивом UCI не проводилось.
- [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api). JSON с почасовым реанализом ERA5 для точки 39,9° с. ш., 116,4° в. д. Это расчётная погода для города, а не показания каждой станции. Атрибуция: Open-Meteo и Copernicus ERA5, данные CC BY 4.0.
- [World Bank API](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures). Исходные XML для Китая, Республики Корея и Монголии за 2013-2015 годы. Показатели: годовая экспозиция PM2.5, население и доля городского населения. [Описание PM2.5](https://databank.worldbank.org/metadataglossary/world-development-indicators/series/EN.ATM.PM25.MC.M3). Данные WDI CC BY 4.0, условия конкретного показателя приведены у источника.
- [Europe PMC REST API](https://europepmc.org/RestfulWebService). Восемь научных аннотаций о Пекине и загрязнении воздуха с годом публикации 2013. Аннотации обрабатываются как свободный текст. Права на отдельные публикации сохраняются у правообладателей; ссылки на статьи включены в таблицу документов.


## Как сформирована выборка

Из каждого CSV взяты первые 48 строк, всего 96. Погодный запрос ограничен двумя датами и даёт 48 часов.
Запрос XML содержит три страны, три года и три показателя: 27 наблюдений плюс три записи справочника стран.
Для текстов задан размер страницы восемь; следующая страница не запрашивается.

Встроенный снимок содержит исходный XML, исходный JSON и строки CSV. Изменение формата здесь не выдаётся
за дополнительный источник. Декодирование сжатого снимка только восстанавливает реальные ответы.


In [4]:
source_plan = pd.DataFrame([
    ('CSV UCI', counts['CSV'], 'station, wd, daypart', 'PM2.5, PM10, NO2; также погода',
     'daypart получен из часа; исходные station и wd сохранены'),
    ('XML World Bank', counts['XML'], 'country, region, income_group_current',
     'annual_pm25, population, urban_pct', 'После разворота: девять строк страна–год; группа дохода текущая'),
    ('JSON Open-Meteo', counts['JSON API'], 'city, weather_group, wind_sector',
     'temperature_2m, wind_speed_10m, relative_humidity_2m',
     'Группа погоды и сектор ветра вычислены; город постоянен'),
    ('Тексты Europe PMC', counts['Тексты'], 'journal, topic, publication_type; также language',
     'char_count, word_count, author_count', 'Объёмы текста вычислены; число авторов взято из метаданных'),
], columns=['Источник', 'Исходных записей', 'Категориальные аспекты', 'Числовые характеристики', 'Уточнение'])
display(source_plan)
assert source_plan['Исходных записей'].le(100).all()


,Источник,Исходных записей,Категориальные аспекты,Числовые характеристики,Уточнение
0,CSV UCI,96,"station, wd, daypart","PM2.5, PM10, NO2; также погода",daypart получен из часа; исходные station и wd сохранены
1,XML World Bank,30,"country, region, income_group_current","annual_pm25, population, urban_pct",После разворота: девять строк страна–год; группа дохода текущая
2,JSON Open-Meteo,48,"city, weather_group, wind_sector","temperature_2m, wind_speed_10m, relative_humidity_2m",Группа погоды и сектор ветра вычислены; город постоянен
3,Тексты Europe PMC,8,"journal, topic, publication_type; также language","char_count, word_count, author_count",Объёмы текста вычислены; число авторов взято из метаданных


Три категориальных аспекта не обязаны меняться в маленькой выборке. Здесь город, язык и регион могут быть постоянными.
Производные категории и числа указаны явно; длина аннотации характеризует текст, а не загрязнение воздуха.


In [5]:
print('CSV: станция, время и концентрации')
display(air[['station', 'timestamp', 'PM2.5', 'PM10', 'NO2', 'wd', 'WSPM']].head(5))
print('JSON API: погода в городе')
display(weather.head(3))
print('XML: годовой страновой контекст')
display(wb.head(3))
print('Неструктурированные тексты: ссылки и объём')
display(docs[['doc_id', 'title', 'journal', 'word_count', 'author_count', 'url']].head(3))


CSV: станция, время и концентрации


,station,timestamp,PM2.5,PM10,NO2,wd,WSPM
0,Aotizhongxin,2013-03-01 00:00:00+08:00,4,4,7.0,NNW,4.4
1,Aotizhongxin,2013-03-01 01:00:00+08:00,8,8,7.0,N,4.7
2,Aotizhongxin,2013-03-01 02:00:00+08:00,7,7,10.0,NNW,5.6
3,Aotizhongxin,2013-03-01 03:00:00+08:00,6,6,11.0,NW,3.1
4,Aotizhongxin,2013-03-01 04:00:00+08:00,3,3,12.0,N,2.0


JSON API: погода в городе


,timestamp,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m,wind_direction_10m,weather_code,city,weather_group,wind_sector
0,2013-03-01 00:00:00+08:00,-0.6,25,1028.3,0.0,4.65,332,3,Beijing,Облачно,NW
1,2013-03-01 01:00:00+08:00,-1.2,24,1028.8,0.0,4.98,338,3,Beijing,Облачно,N
2,2013-03-01 02:00:00+08:00,-1.2,23,1029.6,0.0,4.31,338,1,Beijing,Ясно,N


XML: годовой страновой контекст


,country_code,country,year,annual_pm25,population,urban_pct,region,income_group_current
0,CHN,China,2013,51.750,1.363e+09,54.49,East Asia & Pacific,Upper middle income
1,CHN,China,2014,52.510,1.372e+09,55.75,East Asia & Pacific,Upper middle income
2,CHN,China,2015,49.473,1.380e+09,57.33,East Asia & Pacific,Upper middle income


Неструктурированные тексты: ссылки и объём


,doc_id,title,journal,word_count,author_count,url
0,MED:23320082,Estimation of citywide air pollution in Beijing.,PloS one,200,5,https://europepmc.org/article/MED/23320082
1,MED:23321859,Malondialdehyde in exhaled breath condensate and urine as a biomarker o...,Journal of exposure science & environmental epidemiology,267,14,https://europepmc.org/article/MED/23321859
2,MED:23358615,Changes of plasma vWF level in response to the improvement of air quali...,Annals of hematology,296,12,https://europepmc.org/article/MED/23358615


In [6]:
provenance = pd.DataFrame(json.loads(RAW['provenance.json']))
file_integrity = pd.DataFrame([
    {'Файл снимка': name, 'Байт UTF-8': len(text.encode('utf-8')),
     'SHA256': hashlib.sha256(text.encode('utf-8')).hexdigest()}
    for name, text in RAW.items() if name != 'provenance.json'
])
display(provenance)
display(file_integrity)


,url,bytes_saved,retrieved_utc
0,https://raw.githubusercontent.com/KM-drago/Beijing-Multi-Site-Air-Quali...,3759,2026-10-26T19:16:28.622194+00:00
1,https://raw.githubusercontent.com/KM-drago/Beijing-Multi-Site-Air-Quali...,3580,2026-10-26T19:16:29.036342+00:00
2,https://archive-api.open-meteo.com/v1/archive?latitude=39.9&longitude=1...,2904,2026-10-26T19:16:29.398899+00:00
3,https://api.worldbank.org/v2/country/CHN;KOR;MNG/indicator/EN.ATM.PM25....,8472,2026-10-26T19:16:30.007742+00:00
4,https://api.worldbank.org/v2/country/CHN;KOR;MNG?per_page=100,1620,2026-10-26T19:16:30.861214+00:00
5,https://www.ebi.ac.uk/europepmc/webservices/rest/search?query=TITLE_ABS...,65982,2026-10-26T19:16:31.668756+00:00


,Файл снимка,Байт UTF-8,SHA256
0,abstracts.json,65982,cfb1f99515d26d0d0524649dcf2d3ea3186f3ef4b85e675a4cf245eaf6b48826
1,Aotizhongxin.csv,3759,c1ccd0b69192f391c59c816cd319c81b2a7432a1c0738d481ed39fa035c986d3
2,Changping.csv,3580,93d0b7865a22761bc4ab4b5a8a9916ab44ccd9c7e19c77f8111f49a1cbf16dd2
3,countries.xml,1620,8b2bf14ef3bc9a2c0b07bb3f3e8560290e1cf2bd31cb5e1d67ffa4140a234d94
4,indicators.xml,8472,733ecad5da9b965c0a6ff43e08d220402b85ddfe8a52c8ea0fd73c3ac2685c31
5,weather.json,2904,453eba5e75ebbb701a4999703d64eabb8aa9520c3b6e16f164ae2c837fdc17a2


## Повторная загрузка

По умолчанию сеть не используется. Для обновления измените REFRESH на True в следующей ячейке.
Запросы сохраняют прежние границы; полный архив UCI не скачивается.
Ошибка сети прерывает обновление, вместо подстановки вымышленных данных.


In [7]:
import urllib.request
import urllib.parse
from datetime import datetime, timezone

def fetch_live():
    fresh, log = {}, []
    def fetch(name, url, params=None, csv_rows=None):
        if params:
            url += '?' + urllib.parse.urlencode(params)
        request = urllib.request.Request(url, headers={'User-Agent': 'AirQualityCoursework/1.0'})
        with urllib.request.urlopen(request, timeout=60) as response:
            if csv_rows is not None:
                # Читаются заголовок и только 48 строк станции. Остальная часть CSV не загружается.
                reader = io.TextIOWrapper(response, encoding='utf-8-sig')
                body = ''.join(reader.readline() for _ in range(csv_rows + 1))
            else:
                data = response.read(2_000_001)
                if len(data) > 2_000_000:
                    raise ValueError('Ответ больше 2 МБ, обновление остановлено')
                body = data.decode('utf-8-sig')
        fresh[name] = body
        log.append({'url': url, 'bytes_saved': len(body.encode('utf-8')),
                    'retrieved_utc': datetime.now(timezone.utc).isoformat()})

    for station in ['Aotizhongxin', 'Changping']:
        fetch(f'{station}.csv',
              f'https://raw.githubusercontent.com/KM-drago/Beijing-Multi-Site-Air-Quality-Data/main/PRSA_Data_{station}_20130301-20170228.csv',
              csv_rows=48)
    fetch('weather.json', 'https://archive-api.open-meteo.com/v1/archive', {
        'latitude': 39.9, 'longitude': 116.4, 'start_date': '2013-03-01', 'end_date': '2013-03-02',
        'hourly': 'temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m,wind_direction_10m,weather_code',
        'timezone': 'Asia/Shanghai', 'wind_speed_unit': 'ms', 'models': 'era5',
    })
    fetch('indicators.xml', 'https://api.worldbank.org/v2/country/CHN;KOR;MNG/indicator/EN.ATM.PM25.MC.M3;SP.POP.TOTL;SP.URB.TOTL.IN.ZS',
          {'source': 2, 'date': '2013:2015', 'per_page': 100})
    fetch('countries.xml', 'https://api.worldbank.org/v2/country/CHN;KOR;MNG', {'per_page': 100})
    fetch('abstracts.json', 'https://www.ebi.ac.uk/europepmc/webservices/rest/search', {
        'query': 'TITLE_ABS:"Beijing" AND TITLE_ABS:"air pollution" AND FIRST_PDATE:[2013-01-01 TO 2013-12-31] AND HAS_ABSTRACT:Y',
        'format': 'json', 'resultType': 'core', 'pageSize': 8, 'sort': 'FIRST_PDATE_D asc',
    })
    fresh['provenance.json'] = json.dumps(log, ensure_ascii=False)
    # Проверка всех лимитов до замены рабочего снимка. При ошибке RAW остаётся прежним.
    read_sources(fresh)
    return fresh

REFRESH = False
if REFRESH:
    with measured('Обновление источников по сети'):
        RAW = fetch_live()
        air, weather, wb, docs, wb_long, countries, counts = read_sources(RAW)
    print('Обновление завершено:', counts)
else:
    print('Использован встроенный снимок. Для повторного запроса задайте REFRESH = True.')


Использован встроенный снимок. Для повторного запроса задайте REFRESH = True.


In [8]:
print('Размеры после загрузки:', counts)
display(pd.DataFrame(TIMINGS))


Размеры после загрузки: {'CSV': 96, 'JSON API': 48, 'XML': 30, 'Тексты': 8}


,Этап,"Время, мс"
0,"Чтение CSV, JSON, XML и текстов",16.944


## Результат пункта 2.3

Найдены и загружены четыре требуемых типа источников. Сохранены адреса запросов и время получения.
Основное исследование опирается на 96 станционных измерений. Остальные источники дополняют их погодой,
страновым фоном и содержанием публикаций.
